In [2]:
import os
os.chdir(r"Q:/sachuriga/Sachuriga_Python/quattrocolo-nwb4fp/src")


import pandas as pd
from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap_ax,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch,calculate_spatial_coherence,calculate_spatial_stability,coherence
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd
# df = pd.read_csv(rf'{ephys_path}\{animal}\{animal}_{day}_unitmatchResults\MatchTable.csv')
# df = pd.read_csv(r'S:\Sachuriga\Ephys_Recording\CR_CA1/65165/65165_2023-07-10_unitmatchResults/MatchTable.csv')


Q:\sachuriga\Sachuriga_Python\quattrocolo-nwb4fp\src\nwb4fp\analyses\examples\tracking_plot.py:5: DeprecationWarning: Please import `center_of_mass` from the `scipy.ndimage` namespace; the `scipy.ndimage.measurements` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.measurements import center_of_mass


In [5]:
def plot_place_cells(npdata,unit_num,ax=None,plot=False):
    ## Load data
    pos_cord = load_speed_fromNWB(npdata['XY_mid_brain'])

    ## filter speed
    raw_pos,combined_array, mask,speeds,smoothed_speed,filtered_speed = pos2speed(pos_cord[:,0], # times
                                pos_cord[:,1], # x
                                pos_cord[:,2], # y
                                filter_speed=True, 
                                min_speed = 0.05)

    ## filter spikes with speed
    raw_pos=combined_array
    # ## filter spikes with speed
    # spk = speed_filtered_spikes(spikes_time,
    #                             pos_cord[:,0], # times
    #                             mask)
    #for i in range(40):
    spikes_time = load_units_fromNWB(npdata['units'], unit_num = unit_num)
    temp = npdata['units']['cell Type'][unit_num]
    temp2 = npdata['units']['functional Cell Type'][unit_num]
    temp3 = npdata['units']['unit_quality'][unit_num]

    spk = speed_filtered_spikes(spikes_time,
                                raw_pos[:,0])
    time_stemp = pos_cord[:,0]
    rate_map = plot_ratemap_ax(raw_pos[:,1], # x
                raw_pos[:,2], # y
                raw_pos[:,0], # times
                spikes_time ,
                box_size=[1.0, 1.0], 
                bin_size=0.05,
                smoothing=0.05,ax=ax,plot=False)

    x_input = npdata['units']['x'][unit_num]
    y_input = npdata['units']['y'][unit_num]

    return rate_map, temp,temp2,temp3

from pathlib import Path

def find_nwb_files(base_folder, animal, day):
    # 将基目录转换为 Path 对象
    base_folder = Path(base_folder)
    # 构造文件名前缀，例如 "65588_2024-03-06_"
    prefix = f"{animal}_{day}_"
    # 定义文件后缀
    suffix = "phy_k_manual.nwb"
    # 构造 glob 模式，* 表示中间可以有任意字符
    pattern = f"{prefix}*{suffix}"
    # 使用 rglob 递归搜索匹配模式的文件，并确保只包含文件（排除目录）
    matching_files = [str(p) for p in base_folder.rglob(pattern) if p.is_file()]
    # 返回匹配的文件路径列表
    return matching_files

def find_folders(directory):
    matching_folders = []
    for folder_name in os.listdir(directory):
        if os.path.isdir(os.path.join(directory, folder_name)) and folder_name.endswith('_unitmatchResults'):
            matching_folders.append(folder_name)
    return matching_folders

In [10]:
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

new_df = pairs_df[(pairs_df['cell_type1']=='Pyramidal cells')&(pairs_df['cell_type2']=='Pyramidal cells')]
new_df

,Session1,Session2,ID1,ID2,cell_type1,cell_type2,quality1,quality2,functional cell type1,functional cell type2,Corr,identifier,animal_id
0,1.0,2.0,1.0,10.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,Place cell,0.005448,1,65165
1,2.0,3.0,10.0,1.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,Place cell,0.053811,1,65165
2,1.0,3.0,1.0,1.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,Place cell,0.123940,1,65165
3,1.0,2.0,1.0,10.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,Place cell,0.005448,2,65165
4,2.0,3.0,10.0,9.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,Place cell,-0.125158,2,65165
5,1.0,3.0,1.0,9.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,Place cell,-0.109942,2,65165
6,1.0,2.0,12.0,17.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,normal_py,0.695505,3,65165
7,2.0,3.0,17.0,16.0,Pyramidal cells,Pyramidal cells,good,good,normal_py,normal_py,-0.007343,3,65165
8,1.0,3.0,12.0,16.0,Pyramidal cells,Pyramidal cells,good,good,Place cell,normal_py,-0.092013,3,65165
10,1.0,2.0,14.0,9.0,Pyramidal cells,Pyramidal cells,good,good,normal_py,normal_py,-0.184868,7,65165


In [35]:
import pandas as pd
# control_ids = ['65165', '65091', '63383', '66539', '65622']
# exp_ids = ['65588', '63385', '66538', '66537', '66922']

control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

df = pairs_df[(pairs_df['cell_type1']=='Pyramidal cells')&(pairs_df['cell_type2']=='Pyramidal cells')]

# Assuming your dataframe is called 'df'
# Step 1: Split into control and experimental groups
control_df = df[df['animal_id'].isin(control_ids)]
exp_df = df[df['animal_id'].isin(exp_ids)]

# Step 2: Define function to create session groups
def create_session_groups(df):
    # Group 1: Session1=1.0 and Session2=2.0
    group1 = df[(df['Session1'] == 1.0) & (df['Session2'] == 2.0)]
    
    # Group 2: Session1=2.0 and Session2=3.0
    group2 = df[(df['Session1'] == 2.0) & (df['Session2'] == 3.0)]
    
    # Group 3: Session1=1.0 and Session2=3.0
    group3 = df[(df['Session1'] == 1.0) & (df['Session2'] == 3.0)]
    
    return group1, group2, group3

# Step 3: Apply grouping to both control and experimental dataframes
control_g1, control_g2, control_g3 = create_session_groups(control_df)
exp_g1, exp_g2, exp_g3 = create_session_groups(exp_df)

# Step 4: Calculate mean 'Corr' values for each group
results = {
    'Control': {
        'Group1 (S1=1.0, S2=2.0)': control_g1['Corr'].mean(),
        'Group2 (S1=2.0, S2=3.0)': control_g2['Corr'].mean(),
        'Group3 (S1=1.0, S2=3.0)': control_g3['Corr'].mean()
    },
    'Experimental': {
        'Group1 (S1=1.0, S2=2.0)': exp_g1['Corr'].mean(),
        'Group2 (S1=2.0, S2=3.0)': exp_g2['Corr'].mean(),
        'Group3 (S1=1.0, S2=3.0)': exp_g3['Corr'].mean()
    }
}

# Print results
for group_type, groups in results.items():
    print(f"\n{group_type} Group Means for 'Corr':")
    for group_name, mean_value in groups.items():
        print(f"{group_name}: {mean_value:.2f}")


Control Group Means for 'Corr':
Group1 (S1=1.0, S2=2.0): 0.11
Group2 (S1=2.0, S2=3.0): 0.05
Group3 (S1=1.0, S2=3.0): 0.55

Experimental Group Means for 'Corr':
Group1 (S1=1.0, S2=2.0): 0.21
Group2 (S1=2.0, S2=3.0): 0.02
Group3 (S1=1.0, S2=3.0): 0.48


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

# [Previous code for data preparation, statistical tests, and printing results remains unchanged]

# 1. Violinplot with Wilcoxon annotations (replacing boxplot)
plt.figure(figsize=(12, 6))
plot_data['Group_Condition'] = plot_data['Group'] + '-' + plot_data['Condition']
ax = sns.violinplot(x='Group', y='Corr', hue='Group_Condition', data=plot_data, 
                    palette=palette, split=False, inner='quartile')
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles[:2], ['Control', 'Experimental'], title='Condition')
plt.title('Violinplot of Correlation Values: Control vs Experimental')
plt.ylabel('Correlation')

# Add Wilcoxon significance annotations
for i, group in enumerate(['Group1', 'Group2', 'Group3']):
    if wilcoxon_results[group]['p_value'] < 0.05:
        y_max = plot_data[plot_data['Group'] == group]['Corr'].max() * 1.1
        plt.text(i, y_max, '*', ha='center', va='bottom', fontsize=12, color='black')

plt.show()

# 2. Histogram (PDF)
plt.figure(figsize=(15, 5))
for i, group in enumerate(['Group1', 'Group2', 'Group3']):
    plt.subplot(1, 3, i+1)
    control_data = plot_data[(plot_data['Group'] == group) & 
                           (plot_data['Condition'] == 'Control')]['Corr']
    if len(control_data) > 0:
        sns.histplot(data=control_data, stat='density', color=control_colors[i], 
                    label='Control', alpha=0.5)
    exp_data = plot_data[(plot_data['Group'] == group) & 
                        (plot_data['Condition'] == 'Experimental')]['Corr']
    if len(exp_data) > 0:
        sns.histplot(data=exp_data, stat='density', color=exp_colors[i], 
                    label='Experimental', alpha=0.5)
    plt.title(f'{group}')
    plt.xlabel('Correlation')
    plt.ylabel('Density')
    plt.legend()
plt.tight_layout()
plt.show()

# 3. CDF with KS and Binomial test annotations
plt.figure(figsize=(15, 5))
for i, group in enumerate(['Group1', 'Group2', 'Group3']):
    plt.subplot(1, 3, i+1)
    control_data = plot_data[(plot_data['Group'] == group) & 
                           (plot_data['Condition'] == 'Control')]['Corr']
    if len(control_data) > 0:
        sns.ecdfplot(data=control_data, color=control_colors[i], 
                    label='Control')
    exp_data = plot_data[(plot_data['Group'] == group) & 
                        (plot_data['Condition'] == 'Experimental')]['Corr']
    if len(exp_data) > 0:
        sns.ecdfplot(data=exp_data, color=exp_colors[i], 
                    label='Experimental')
    plt.title(f'{group}')
    plt.xlabel('Correlation')
    plt.ylabel('Cumulative Probability')
    plt.legend()
    
    # Add KS test significance annotation
    if ks_results[group]['p_value'] < 0.05:
        plt.text(0.5, 0.1, 'KS p < 0.05', ha='center', va='center', 
                 fontsize=10, color='red', transform=plt.gca().transAxes)
    
    # Add Binomial test significance annotation
    if binom_results[group]['p_value'] < 0.05:
        plt.text(0.5, 0.05, 'Binom p < 0.05', ha='center', va='center', 
                 fontsize=10, color='purple', transform=plt.gca().transAxes)
plt.tight_layout()
plt.show()

NameError: name 'plot_data' is not defined

<Figure size 1200x600 with 0 Axes>

In [ ]:

animal = "65588"
#day="2024-03-04"
base_folder="S:/Sachuriga/nwb/test4neo/"
base_recordings = r"S:\Sachuriga\Ephys_Recording\CR_CA1/"
animals = ['65165', '65091', '63383', '66539', '65622','65588', '63385', '66538', '66537', '66922']

pairs_df = pd.DataFrame(columns=['Session1', 'Session2', 'ID1', 'ID2', 
                                'cell_type1','cell_type2', 'quality1','quality2', 'functional cell type1','functional cell type2', 
                                'Corr','identifier','animal_id'])

global identifier 

identifier = 0
error_log=[]
for animal in animals:
    search_path_head = fr"{base_recordings}/{animal}"
    files = find_folders(search_path_head)
    for f in files:
        print(fr"rf'S:\Sachuriga/Ephys_Recording/CR_CA1/{animal}/{f}/MatchTable.csv'")
        df = pd.read_csv(rf'S:\Sachuriga/Ephys_Recording/CR_CA1/{animal}/{f}/MatchTable.csv')
        try:
            pairs_df = collect_remap(base_folder,animal,f.split("_")[1],pairs_df,df,identifier)
        except Exception as e:
            error_log.append(f)

In [10]:
 files

[]

In [7]:

def collect_remap(base_folder,animal,day,pairs_df,df,identifier,plot=False):
    import pandas as pd
    #df = pd.read_csv(rf'S:\Sachuriga/Ephys_Recording/CR_CA1/{animal}/{animal}_{day}_unitmatchResults/MatchTable.csv')
    # df1 = df[df['Matches']==1]
    # df1 

    df2 = df[(df['UM Probabilities']>0.8)&(df['Matches']==1)]
    #df2 = df[df['Matches']==1]

    # for i in range(4):
    #     temp_df.append(df2[(df2['RecSes 1'] == i) & (df2['RecSes 2'] !=i)])
    # df_s=pd.concat(temp_df, ignore_index=True)
    all_temp_rows = []
    temp_all_temp_rows=[]
    temp_df=[]

    i=1
    temp_df.append(df2[(df2['RecSes 1'] == i) & (df2['RecSes 2'] !=i)])
    df_s=pd.concat(temp_df, ignore_index=True)
    match_sessions = np.unique(df_s['RecSes 2'])
    for s in match_sessions:
        temp_df = df_s[df_s['RecSes 2']==s]
        for i in np.unique(temp_df['ID2']):
            temp_id_df = temp_df[temp_df['ID2']==i].sort_values(by='UM Probabilities',ascending=False)
            temp_row = temp_id_df.head(1)
            temp_all_temp_rows.append(temp_row)
        
        temp_df = pd.concat(temp_all_temp_rows, ignore_index=True)
        temp_df = temp_df[(temp_df['RecSes 2']==s)&(temp_df['RecSes 1']==1)]

        for i in np.unique(temp_df['ID1']):
            temp_id_df = temp_df[temp_df['ID1']==i].sort_values(by='UM Probabilities',ascending=False)
            temp_row = temp_id_df.head(1)
            all_temp_rows.append(temp_row)

    #     temp_df = pd.concat(temp_all_temp_rows, ignore_index=True)
    #     for i in np.unique(temp_df['ID2']):
    #         temp_id_df = temp_df[temp_df['ID2']==i].sort_values(by='UM Probabilities',ascending=False)
    #         temp_row = temp_id_df.head(1)
    #         all_temp_rows.append(temp_row)

    temp_all_temp_rows=[]
    if len(match_sessions)>=2:
        print("here")
        df_s2 = df2[(df2['RecSes 1'] == match_sessions[0]) & (df2['RecSes 2'] == match_sessions[1])]
        for j in np.unique(df_s2['ID1']):
            temp_id_dfs2 = df_s2[df_s2['ID1']==j].sort_values(by='UM Probabilities',ascending=False)
            temp_rows = temp_id_dfs2.head(1)
            temp_all_temp_rows.append(temp_rows)

        temp_df = pd.concat(temp_all_temp_rows, ignore_index=True)
        for j in np.unique(df_s2['ID2']):
            temp_id_dfs2 = df_s2[df_s2['ID2']==j].sort_values(by='UM Probabilities',ascending=False)
            temp_rows = temp_id_dfs2.head(1)
            all_temp_rows.append(temp_rows)

    match_sessions = pd.concat(all_temp_rows, ignore_index=True)
    new_df = match_sessions


    import pandas as pd
    pd.set_option('display.max_rows', None)
    np.set_printoptions(threshold=np.inf)

    session_data = []
    files = find_nwb_files(base_folder, animal, day)

    for file in files:
        session_data.append(nap.load_file(file))
        print(file)
    pairs_df = plot_remap(new_df,session_data,pairs_df,identifier,animal,plot=False)
    return pairs_df

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pynapple.core.metadata_class")

def plot_remap(new_df,session_data,pairs_df,identifier,animal,plot=False):
    # 假设数据结构：
    # new_df: 包含列 ['RecSes 1', 'RecSes 2', 'ID1', 'ID2', 'UM Probabilities']
    # session_data: {1: 数据1, 2: 数据2, 3: 数据3}

    # pairs_df = pd.DataFrame(columns=['Session1', 'Session2', 'ID1', 'ID2', 
    #                                 'cell_type', 'quality', 'functional cell type', 
    #                                 'Corr', 'Coherence1', 'Coherence2'])
    
    pair_data = []
    if len(session_data)==3:
        # Step 1: 筛选 Session 1 → Session 2 和 Session 2 → Session 3 的配对
        A = new_df[(new_df['RecSes 1'] == 1) & (new_df['RecSes 2'] == 2)]
        B = new_df[(new_df['RecSes 1'] == 2) & (new_df['RecSes 2'] == 3)]

        # Step 2: 通过合并 A 和 B 找到三联体
        merged = pd.merge(A, B, left_on='ID2', right_on='ID1', suffixes=('_A', '_B'))

        # Step 3: 绘制三联体并记录配对
        triplet_pairs = set()  # 存储三联体中的配对

        for index, row in merged.iterrows():
            identifier += 1
            # 三联体的单元 ID
            id1_s1 = row['ID1_A']  # Session 1
            id2_s2 = row['ID2_A']  # Session 2
            id2_s3 = row['ID2_B']  # Session 3
            
            # 获取会话数据
            data1 = session_data[0]
            data2 = session_data[1]
            data3 = session_data[2]
            
            # 创建 1x3 subplot
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
            
            # 绘制 Session 1
            rate_map1,t1,t12,t13 = plot_place_cells(data1, unit_num=id1_s1, ax=ax1)
            coherence1 = coherence(rate_map1)
            ax1.set_title(f'RecSes 1,{t1}, Unit {id1_s1}\nCoherence: {coherence1:.3f}, quality: {t13}')
            
            # 绘制 Session 2
            rate_map2,t2,t22,t23 = plot_place_cells(data2, unit_num=id2_s2, ax=ax2)
            coherence2 = coherence(rate_map2)
            ax2.set_title(f'RecSes 2,{t2}, Unit {id2_s2}\nCoherence: {coherence2:.3f}, quality: {t23}')
            
            # 绘制 Session 3
            rate_map3,t3,t32,t33 = plot_place_cells(data3, unit_num=id2_s3, ax=ax3)
            coherence3 = coherence(rate_map3)
            ax3.set_title(f'RecSes 3,{t2}, Unit {id2_s3}\nCoherence: {coherence3:.3f}, quality: {t33}')
            
            # 计算相关性
            corr12 = calculate_spatial_stability(rate_map1, rate_map2)
            corr23 = calculate_spatial_stability(rate_map2, rate_map3)
            corr13 = calculate_spatial_stability(rate_map1, rate_map3)
            # 添加标题
            fig.suptitle(f'Triplet: (1, {id1_s1}) - (2, {id2_s2}) - (3, {id2_s3})\n'
                        f'Corr 1-2: {corr12:.6f}, Corr 2-3: {corr23:.6f},Corr 1-3: {corr13:.6f}')
            
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()
            
            # 记录三联体中的配对
            triplet_pairs.add((1, 2, id1_s1, id2_s2))  # Session 1 → Session 2
            pair_data.append({
                    'Session1': 1, 'Session2': 2, 'ID1': id1_s1, 'ID2':id2_s2,
                    'cell_type1':t1, 'cell_type2':t2, 'quality1':t13, 'quality2':t23,'functional cell type1':t12,'functional cell type2':t22,  
                    'Corr':corr12,'identifier':identifier,'animal_id':animal})

            triplet_pairs.add((2, 3, id2_s2, id2_s3))  # Session 2 → Session 3
            pair_data.append({
                    'Session1': 2, 'Session2': 3, 'ID1': id2_s2, 'ID2':id2_s3,
                    'cell_type1':t2, 'cell_type2':t3, 'quality1':t23, 'quality2':t33,'functional cell type1':t22,'functional cell type2':t32, 
                    'Corr':corr23,'identifier':identifier,'animal_id':animal})
            triplet_pairs.add((1, 3, id1_s1, id2_s3)) 
            pair_data.append({
                    'Session1': 1, 'Session2': 3, 'ID1': id1_s1, 'ID2':id2_s3,
                    'cell_type1':t1, 'cell_type2':t3, 'quality1':t13, 'quality2':t33,'functional cell type1':t12,'functional cell type2':t32, 
                    'Corr':corr13,'identifier':identifier,'animal_id':animal})
            print(triplet_pairs)
        # Step 4: 绘制二联体，跳过在三联体中出现过的配对
        for index, row in new_df.iterrows():
            identifier += 1
            rec_ses1, rec_ses2 = row['RecSes 1'], row['RecSes 2']
            id1, id2 = row['ID1'], row['ID2']
            prob = row['UM Probabilities']
            
            # 检查当前配对是否在三联体中
            current_pair = (rec_ses1, rec_ses2, id1, id2)
            if current_pair in triplet_pairs:
                continue  # 跳过
            print(current_pair)
            # 获取数据
            data1 = session_data[np.int64(rec_ses1)-1]
            data2 = session_data[np.int64(rec_ses2)-1]
            
            # 创建 1x2 subplot
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            
            # 绘制第一个 place cell
            rate_map1,t1,t12,t13 = plot_place_cells(data1, unit_num=id1, ax=ax1)
            coherence1 = coherence(rate_map1)
            ax1.set_title(f'RecSes {rec_ses1},{t1}, Unit {id1}\nCoherence: {coherence1:.3f}, quality: {t13}')
            
            # 绘制第二个 place cell
            rate_map2,t2,t22,t23 = plot_place_cells(data2, unit_num=id2, ax=ax2)
            coherence2 = coherence(rate_map2)
            ax2.set_title(f'RecSes {rec_ses2},{t2}, Unit {id2}\nCoherence: {coherence2:.3f}, quality: {t23}')
            
            # 计算相关性
            correlation = calculate_spatial_stability(rate_map1, rate_map2)
            pair_data.append({
                    'Session1': rec_ses1, 'Session2': rec_ses2, 'ID1': id1, 'ID2':id2,
                    'cell_type1':t1, 'cell_type2':t2, 'quality1':t13, 'quality2':t23,'functional cell type1':t12,'functional cell type2':t22, 
                    'Corr':correlation,'identifier':identifier,'animal_id':animal})
            # 添加标题
            fig.suptitle(f'Pair: ({rec_ses1}, {id1}) - ({rec_ses2}, {id2}), '
                        f'UM Probability: {prob:.6f}, Correlation: {correlation:.6f}')
            
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()
    else:
       
        triplet_pairs = set()
        # Step 4: 绘制二联体，跳过在三联体中出现过的配对
        for index, row in new_df.iterrows():
            identifier += 1
            rec_ses1, rec_ses2 = row['RecSes 1'], row['RecSes 2']
            id1, id2 = row['ID1'], row['ID2']
            prob = row['UM Probabilities']
            
            # 检查当前配对是否在三联体中
            current_pair = (rec_ses1, rec_ses2, id1, id2)
            if current_pair in triplet_pairs:
                continue  # 跳过
            
            # 获取数据
            data1 = session_data[np.int64(rec_ses1)-1]
            data2 = session_data[np.int64(rec_ses2)-1]
            
            # 创建 1x2 subplot
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            
            # 绘制第一个 place cell
            rate_map1,t1,t12,t13 = plot_place_cells(data1, unit_num=id1, ax=ax1)
            coherence1 = coherence(rate_map1)
            ax1.set_title(f'RecSes {rec_ses1},{t1},  Unit {id1}\nCoherence: {coherence1:.3f}, quality: {t13}')
            
            # 绘制第二个 place cell
            rate_map2,t2,t22,t23 = plot_place_cells(data2, unit_num=id2, ax=ax2)
            coherence2 = coherence(rate_map2)
            ax2.set_title(f'RecSes {rec_ses2},{t2},  Unit {id2}\nCoherence: {coherence2:.3f}, quality: {t23}')
            
            # 计算相关性
            correlation = calculate_spatial_stability(rate_map1, rate_map2)
            pair_data.append({
                    'Session1': rec_ses1, 'Session2': rec_ses2, 'ID1': id1, 'ID2':id2,
                    'cell_type1':t1, 'cell_type2':t2, 'quality1':t13, 'quality2':t23,'functional cell type1':t12,'functional cell type2':t22, 
                    'Corr':correlation,'identifier':identifier,'animal_id':animal})
            # 添加标题
            fig.suptitle(f'Pair: ({rec_ses1}, {id1}) - ({rec_ses2}, {id2}), '
                        f'UM Probability: {prob:.6f}, Correlation: {correlation:.6f}')
            
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()
        print("所有三联体和未重复的二联体 place cell 图已生成！")

    return pd.concat([pairs_df, pd.DataFrame(pair_data)], ignore_index=True)